# 练习 3：装饰器 @decorator

**AI 应用场景**

- 统计每个 LLM 调用的耗时（不想改原函数代码）
- 给工具函数自动加"失败重试"
- FastAPI 的 `@app.get("/")` 也是装饰器

**核心语法**

- `@decorator` 写在 def 上面 = Python 自动执行 `func = decorator(func)`
- `wrapper(*args, **kwargs)` 能接住任意参数，转发给原函数
- `@functools.wraps(func)` 保留原函数的名字和文档

In [1]:
import functools
import time
from typing import Callable

def log_time(func: Callable) -> Callable:
    # 装饰器：打印函数执行耗时
    @functools.wraps(func)  # 不加这个，wrapper 会"覆盖"原函数的名字
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)  # 调用原函数
        print(f"⏱️  函数 {func.__name__} 执行耗时: {time.time() - start:.2f}s")
        return result
    return wrapper

print("log_time 已定义 ✅")

log_time 已定义 ✅


## 用法：@log_time 装饰一个"LLM 调用"

In [2]:
@log_time
def call_llm_model() -> str:
    # 模拟一次 LLM 调用（真调用时这里换成 self.client.chat...）
    time.sleep(0.3)
    return "模型回复内容"

result = call_llm_model()
print(f"结果: {result}")
print(f"函数名: {call_llm_model.__name__}（functools.wraps 保留了原名）")

⏱️  函数 call_llm_model 执行耗时: 0.31s
结果: 模型回复内容
函数名: call_llm_model（functools.wraps 保留了原名）


## 带参数的装饰器：失败自动重试

装饰器本身还能再套一层：`@retry(max_attempts=3)` 先用参数生成装饰器，再装饰函数。

In [3]:
def retry(max_attempts: int = 3) -> Callable:
    # 带参数的装饰器：失败自动重试（AI 调用常用）
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  第 {attempt} 次失败: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

@retry(max_attempts=3)
def unstable_api() -> str:
    # 模拟一个偶尔失败的 API（用当前时间模拟随机性）
    if time.time() % 1 > 0.5:
        raise ConnectionError("网络波动")
    return "成功"

try:
    print(f"结果: {unstable_api()}")
except Exception as e:
    print(f"最终失败: {e}（多跑几次，会看到重试过程）")

  第 1 次失败: 网络波动
  第 2 次失败: 网络波动
  第 3 次失败: 网络波动
最终失败: 网络波动（多跑几次，会看到重试过程）


## 💡 小结

`@log_time` 写在 def 上面，Python 等价于帮你执行了：

```python
call_llm_model = log_time(call_llm_model)
```

原函数一行没改，但外面多了一层"包装" —— 这就是**开闭原则**：对扩展开放，对修改关闭。